# Live GPU Parallelism Dashboard — Windows + NVIDIA CUDA

This notebook is a **local-machine companion** to the Fashion-MNIST batch-size notebook.

Its teaching goal is not only to report final timing numbers. It lets students watch the GPU respond while the same network trains with different batch sizes.

The notebook demonstrates:

1. **Actual live GPU utilization**
2. **GPU memory use**
3. **Power draw and temperature**
4. **Samples processed per second**
5. **How utilization and throughput change as batch size grows**
6. **Where increasing batch size stops helping**
7. **A real PyTorch CPU/GPU execution trace**

The model remains a plain MLP. No CNN knowledge is required.

> Important: NVIDIA's “GPU utilization” is the percentage of the recent sampling period during which one or more GPU kernels were executing. It is not literally the percentage of individual CUDA cores currently lit up.

## 0. Environment assumptions

This notebook expects:

- Windows
- an NVIDIA GPU
- a working NVIDIA driver
- a CUDA-enabled PyTorch installation
- Jupyter Notebook or VS Code notebooks
- the notebook kernel running in the same Python environment as PyTorch

Check CUDA first:

```python
import torch
print(torch.cuda.is_available())
```

If this prints `False`, fix the PyTorch/CUDA environment before continuing.

In [19]:
import importlib
import subprocess
import sys


def ensure_package_installed(package_name, import_name=None):
    """
    Ensures a Python package is installed and imported.

    Args:
        package_name: Name used by pip.
        import_name: Name used by import. Defaults to package_name.

    Returns:
        The imported module.
    """
    import_name = import_name or package_name

    try:
        return importlib.import_module(import_name)
    except ImportError:
        print(f"Installing '{package_name}'...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package_name]
        )
        return importlib.import_module(import_name)


# Official Python bindings for NVIDIA's NVML monitoring library.
pynvml = ensure_package_installed(
    "nvidia-ml-py",
    import_name="pynvml",
)

In [20]:
import copy
import gc
import math
import os
import random
import threading
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Use a CUDA-enabled PyTorch environment."
    )

DEVICE = torch.device("cuda")
GPU_INDEX = 0

print("PyTorch:", torch.__version__)
print("CUDA runtime used by PyTorch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(GPU_INDEX))
print(
    "Total GPU memory:",
    f"{torch.cuda.get_device_properties(GPU_INDEX).total_memory / 1024**3:.1f} GB"
)

PyTorch: 2.13.0+cu132
CUDA runtime used by PyTorch: 13.2
GPU: NVIDIA GeForce RTX 3090
Total GPU memory: 24.0 GB


## 1. Connect to NVIDIA's monitoring API

`nvidia-smi` obtains its readings through NVIDIA's Management Library, NVML.

We will use the same monitoring source directly from Python.

Metrics sampled during training:

- GPU utilization
- memory-controller utilization
- used GPU memory
- temperature
- power draw

The sampling interval is intentionally around 200 ms. Very short neural-network steps may finish between samples, so each experiment runs for several seconds.

In [21]:
from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetName,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetTemperature,
    nvmlDeviceGetPowerUsage,
    nvmlDeviceGetPowerManagementLimit,
    NVML_TEMPERATURE_GPU,
    NVMLError,
)

nvmlInit()
NVML_HANDLE = nvmlDeviceGetHandleByIndex(GPU_INDEX)

gpu_name = nvmlDeviceGetName(NVML_HANDLE)
if isinstance(gpu_name, bytes):
    gpu_name = gpu_name.decode("utf-8")

print("NVML connected to:", gpu_name)


def read_gpu_metrics():
    """
    Read one NVML sample.

    GPU utilization means the percentage of the recent sample period
    during which one or more kernels executed.
    """
    utilization = nvmlDeviceGetUtilizationRates(NVML_HANDLE)
    memory = nvmlDeviceGetMemoryInfo(NVML_HANDLE)

    result = {
        "timestamp": time.perf_counter(),
        "gpu_utilization_percent": float(utilization.gpu),
        "memory_controller_percent": float(utilization.memory),
        "memory_used_mb": memory.used / 1024**2,
        "memory_total_mb": memory.total / 1024**2,
    }

    try:
        result["temperature_c"] = float(
            nvmlDeviceGetTemperature(
                NVML_HANDLE,
                NVML_TEMPERATURE_GPU,
            )
        )
    except NVMLError:
        result["temperature_c"] = float("nan")

    try:
        result["power_w"] = nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000
        result["power_limit_w"] = (
            nvmlDeviceGetPowerManagementLimit(NVML_HANDLE) / 1000
        )
    except NVMLError:
        result["power_w"] = float("nan")
        result["power_limit_w"] = float("nan")

    return result


read_gpu_metrics()

NVML connected to: NVIDIA GeForce RTX 3090


{'timestamp': 1534086.7867853,
 'gpu_utilization_percent': 27.0,
 'memory_controller_percent': 4.0,
 'memory_used_mb': 6059.55859375,
 'memory_total_mb': 24576.0,
 'temperature_c': 50.0,
 'power_w': 88.032,
 'power_limit_w': 340.4}

## 2. Create a deliberately compute-heavy MLP

The input has 3,072 values — equivalent to a flattened $32 \times 32$ RGB image.

The network is intentionally wider than necessary:

$$
3072 \rightarrow 4096 \rightarrow 4096 \rightarrow 2048 \rightarrow 10
$$

Synthetic data is generated directly on the GPU. This prevents disk loading and CPU transfer from becoming the main lesson.

In [22]:
INPUT_FEATURES = 3072
NUMBER_OF_CLASSES = 10
SYNTHETIC_DATASET_SIZE = 65_536


class StressMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(INPUT_FEATURES, 4096),
            nn.ReLU(),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, 2048),
            nn.ReLU(),
            nn.Linear(2048, NUMBER_OF_CLASSES),
        )

    def forward(self, x):
        return self.network(x)


# Keep one reusable synthetic dataset on the GPU.
X_SYNTHETIC = torch.randn(
    SYNTHETIC_DATASET_SIZE,
    INPUT_FEATURES,
    device=DEVICE,
)

Y_SYNTHETIC = torch.randint(
    low=0,
    high=NUMBER_OF_CLASSES,
    size=(SYNTHETIC_DATASET_SIZE,),
    device=DEVICE,
)

example_model = StressMLP().to(DEVICE)

parameter_count = sum(
    parameter.numel()
    for parameter in example_model.parameters()
)

print(f"Trainable parameters: {parameter_count:,}")
print("Synthetic data:", X_SYNTHETIC.shape)

Trainable parameters: 37,779,466
Synthetic data: torch.Size([65536, 3072])


## 3. Benchmark engine with background GPU monitoring

For each selected batch size:

1. Create the same model architecture.
2. Warm up CUDA.
3. Start a background NVML monitor.
4. Run full training steps for several seconds.
5. Stop monitoring.
6. Calculate throughput and summary statistics.

Each training step includes:

```text
forward → loss → backward → optimizer step
```

In [23]:
@dataclass
class RunSummary:
    batch_size: int
    steps: int
    samples_processed: int
    elapsed_seconds: float
    milliseconds_per_step: float
    samples_per_second: float
    mean_gpu_utilization_percent: float
    peak_gpu_utilization_percent: float
    mean_memory_controller_percent: float
    peak_memory_used_mb: float
    mean_power_w: float
    peak_temperature_c: float
    status: str


def synchronize_cuda():
    torch.cuda.synchronize()


def monitor_gpu(stop_event, samples, interval_seconds=0.2):
    """
    Sample NVML until stop_event is set.
    """
    while not stop_event.is_set():
        try:
            samples.append(read_gpu_metrics())
        except NVMLError:
            pass

        stop_event.wait(interval_seconds)


def run_training_experiment(
    batch_size,
    duration_seconds=8.0,
    warmup_steps=8,
    learning_rate=0.01,
):
    """
    Run complete training steps for approximately duration_seconds.

    Returns:
        summary: RunSummary
        samples_df: one row per NVML sample
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = StressMLP().to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=learning_rate,
    )
    loss_function = nn.CrossEntropyLoss()

    sample_count = len(X_SYNTHETIC)
    monitor_samples = []
    stop_event = threading.Event()
    monitor_thread = None

    try:
        # Warm-up is excluded from measurements.
        for step in range(warmup_steps):
            start = (step * batch_size) % sample_count
            indices = (
                torch.arange(
                    start,
                    start + batch_size,
                    device=DEVICE,
                )
                % sample_count
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(X_SYNTHETIC[indices])
            loss = loss_function(logits, Y_SYNTHETIC[indices])
            loss.backward()
            optimizer.step()

        synchronize_cuda()

        monitor_thread = threading.Thread(
            target=monitor_gpu,
            args=(stop_event, monitor_samples),
            daemon=True,
        )
        monitor_thread.start()

        start_time = time.perf_counter()
        steps = 0

        while True:
            start = ((steps + warmup_steps) * batch_size) % sample_count
            indices = (
                torch.arange(
                    start,
                    start + batch_size,
                    device=DEVICE,
                )
                % sample_count
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(X_SYNTHETIC[indices])
            loss = loss_function(logits, Y_SYNTHETIC[indices])
            loss.backward()
            optimizer.step()

            steps += 1

            # Check elapsed time only after completed CUDA work.
            if steps % 4 == 0:
                synchronize_cuda()

                if time.perf_counter() - start_time >= duration_seconds:
                    break

        synchronize_cuda()
        elapsed_seconds = time.perf_counter() - start_time

        stop_event.set()
        monitor_thread.join(timeout=2)

        samples_processed = steps * batch_size
        milliseconds_per_step = elapsed_seconds / steps * 1000
        samples_per_second = samples_processed / elapsed_seconds

        samples_df = pd.DataFrame(monitor_samples)

        if samples_df.empty:
            mean_utilization = float("nan")
            peak_utilization = float("nan")
            mean_memory_controller = float("nan")
            mean_power = float("nan")
            peak_temperature = float("nan")
        else:
            mean_utilization = samples_df[
                "gpu_utilization_percent"
            ].mean()

            peak_utilization = samples_df[
                "gpu_utilization_percent"
            ].max()

            mean_memory_controller = samples_df[
                "memory_controller_percent"
            ].mean()

            mean_power = samples_df["power_w"].mean()
            peak_temperature = samples_df["temperature_c"].max()

        summary = RunSummary(
            batch_size=batch_size,
            steps=steps,
            samples_processed=samples_processed,
            elapsed_seconds=elapsed_seconds,
            milliseconds_per_step=milliseconds_per_step,
            samples_per_second=samples_per_second,
            mean_gpu_utilization_percent=mean_utilization,
            peak_gpu_utilization_percent=peak_utilization,
            mean_memory_controller_percent=mean_memory_controller,
            peak_memory_used_mb=(
                torch.cuda.max_memory_allocated() / 1024**2
            ),
            mean_power_w=mean_power,
            peak_temperature_c=peak_temperature,
            status="OK",
        )

        return summary, samples_df

    except torch.cuda.OutOfMemoryError:
        stop_event.set()

        if monitor_thread is not None:
            monitor_thread.join(timeout=2)

        torch.cuda.empty_cache()

        summary = RunSummary(
            batch_size=batch_size,
            steps=0,
            samples_processed=0,
            elapsed_seconds=0,
            milliseconds_per_step=float("nan"),
            samples_per_second=float("nan"),
            mean_gpu_utilization_percent=float("nan"),
            peak_gpu_utilization_percent=float("nan"),
            mean_memory_controller_percent=float("nan"),
            peak_memory_used_mb=float("nan"),
            mean_power_w=float("nan"),
            peak_temperature_c=float("nan"),
            status="Out of memory",
        )

        return summary, pd.DataFrame()

    finally:
        stop_event.set()

        del model
        del optimizer

        gc.collect()
        torch.cuda.empty_cache()

## 4. Interactive classroom dashboard

Choose a batch size and press **Run Experiment**.

The “GPU blocks” are a teaching visualization driven by the measured average utilization. They are **not a literal map of individual CUDA cores**.

Suggested classroom sequence:

```text
Batch 1 → 8 → 32 → 128 → 512 → 2048
```

Before each run, ask students to predict:

- utilization;
- throughput;
- memory use;
- whether doubling the batch will double throughput.

In [24]:
# Shared ipywidgets slider style/layout
SLIDER_STYLE = {"description_width": "160px"}
SLIDER_LAYOUT = widgets.Layout(width="520px")

batch_slider = widgets.SelectionSlider(
    options=[
        1, 2, 4, 8, 16, 32, 64,
        128, 256, 512, 1024, 2048, 4096
    ],
    value=32,
    description="Batch size",
    continuous_update=False,
    style=SLIDER_STYLE,
    layout=SLIDER_LAYOUT,
)

duration_slider = widgets.IntSlider(
    value=8,
    min=4,
    max=20,
    step=1,
    description="Run duration (seconds)",
    continuous_update=False,
    style=SLIDER_STYLE,
    layout=SLIDER_LAYOUT,
)

run_button = widgets.Button(
    description="Run Experiment",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

clear_button = widgets.Button(
    description="Clear Results",
    layout=widgets.Layout(width="140px"),
)

dashboard_output = widgets.Output()

RUN_SUMMARIES = []
RUN_TRACES = {}


def utilization_blocks(utilization, block_count=40):
    """
    Convert measured utilization into a schematic block display.

    This is conceptual—not a physical CUDA-core map.
    """
    if not np.isfinite(utilization):
        active_blocks = 0
    else:
        active_blocks = int(
            round(utilization / 100 * block_count)
        )

    return (
        '<div style="display:grid;'
        f'grid-template-columns:repeat({block_count},1fr);'
        'gap:3px;margin:8px 0 4px;">'
        + "".join(
            (
                '<div style="height:24px;border-radius:3px;'
                'background:#2f6feb;"></div>'
                if index < active_blocks
                else
                '<div style="height:24px;border-radius:3px;'
                'background:#c9ced8;"></div>'
            )
            for index in range(block_count)
        )
        + "</div>"
    )


def metric_card(label, value, note=""):
    return f"""
    <div style="
        border:1px solid #c8ced8;
        border-radius:10px;
        padding:12px;
        background:#ffffff;
        color:#172033;
    ">
        <div style="font-size:12px;color:#667085;">{label}</div>
        <div style="font-size:24px;font-weight:750;">{value}</div>
        <div style="font-size:11px;color:#667085;">{note}</div>
    </div>
    """


def render_dashboard(summary, samples_df):
    if summary.status != "OK":
        display(
            HTML(
                f"""
                <div style="
                    border:2px solid #b42318;
                    border-radius:10px;
                    padding:14px;
                    background:#fff0ee;
                    color:#64120d;
                ">
                    Batch size {summary.batch_size}: {summary.status}
                </div>
                """
            )
        )
        return

    display(
        HTML(
            f"""
            <div style="
                font-family:Arial,sans-serif;
                border:1px solid #c8ced8;
                border-radius:12px;
                padding:14px;
                background:#f7f9fc;
                color:#172033;
            ">
                <div style="
                    display:flex;
                    justify-content:space-between;
                    align-items:end;
                    gap:12px;
                ">
                    <div>
                        <div style="font-size:12px;color:#667085;">
                            Current experiment
                        </div>
                        <div style="font-size:22px;font-weight:800;">
                            Batch size {summary.batch_size}
                        </div>
                    </div>
                    <div style="font-size:12px;color:#667085;">
                        Actual NVML measurements
                    </div>
                </div>

                <div style="margin-top:14px;font-weight:700;">
                    Schematic GPU activity
                </div>

                {utilization_blocks(
                    summary.mean_gpu_utilization_percent
                )}

                <div style="font-size:11px;color:#667085;">
                    Blocks represent measured average utilization conceptually;
                    they are not individual CUDA cores.
                </div>

                <div style="
                    display:grid;
                    grid-template-columns:repeat(3,minmax(0,1fr));
                    gap:10px;
                    margin-top:14px;
                ">
                    {metric_card(
                        "Mean GPU utilization",
                        f"{summary.mean_gpu_utilization_percent:.0f}%",
                        f"Peak {summary.peak_gpu_utilization_percent:.0f}%"
                    )}
                    {metric_card(
                        "Throughput",
                        f"{summary.samples_per_second:,.0f}",
                        "samples per second"
                    )}
                    {metric_card(
                        "Time per step",
                        f"{summary.milliseconds_per_step:.2f} ms",
                        f"{summary.steps} measured steps"
                    )}
                    {metric_card(
                        "Peak allocated memory",
                        f"{summary.peak_memory_used_mb:,.0f} MB",
                        "PyTorch CUDA allocator"
                    )}
                    {metric_card(
                        "Mean power",
                        (
                            f"{summary.mean_power_w:.0f} W"
                            if np.isfinite(summary.mean_power_w)
                            else "Unavailable"
                        ),
                        "NVML reading"
                    )}
                    {metric_card(
                        "Peak temperature",
                        (
                            f"{summary.peak_temperature_c:.0f} °C"
                            if np.isfinite(summary.peak_temperature_c)
                            else "Unavailable"
                        ),
                        "NVML reading"
                    )}
                </div>
            </div>
            """
        )
    )

    if not samples_df.empty:
        relative_time = (
            samples_df["timestamp"] - samples_df["timestamp"].iloc[0]
        )

        plt.figure(figsize=(10, 3.6))
        plt.plot(
            relative_time,
            samples_df["gpu_utilization_percent"],
            label="GPU utilization",
        )
        plt.plot(
            relative_time,
            samples_df["memory_controller_percent"],
            label="Memory-controller utilization",
        )
        plt.xlabel("Time since monitoring began (seconds)")
        plt.ylabel("Utilization (%)")
        plt.ylim(0, 105)
        plt.title(
            f"Live Hardware Trace — Batch {summary.batch_size}"
        )
        plt.grid(True)
        plt.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.2),
            ncol=1,
        )
        plt.tight_layout()
        plt.show()


def render_comparison():
    if not RUN_SUMMARIES:
        return

    comparison_df = pd.DataFrame(
        [asdict(summary) for summary in RUN_SUMMARIES]
    )

    valid = comparison_df[
        comparison_df["status"] == "OK"
    ].copy()

    if valid.empty:
        return

    display(
        valid[
            [
                "batch_size",
                "samples_per_second",
                "mean_gpu_utilization_percent",
                "milliseconds_per_step",
                "peak_memory_used_mb",
                "mean_power_w",
            ]
        ].sort_values("batch_size")
    )

    plt.figure(figsize=(9, 4.2))
    plt.plot(
        valid["batch_size"],
        valid["samples_per_second"],
        marker="o",
    )
    plt.xscale("log", base=2)
    plt.xlabel("Batch size")
    plt.ylabel("Samples per second")
    plt.title("Measured Throughput Curve")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 4.2))
    plt.plot(
        valid["batch_size"],
        valid["mean_gpu_utilization_percent"],
        marker="o",
    )
    plt.xscale("log", base=2)
    plt.xlabel("Batch size")
    plt.ylabel("Mean GPU utilization (%)")
    plt.ylim(0, 105)
    plt.title("Measured GPU Utilization Curve")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def on_run_clicked(_):
    run_button.disabled = True
    clear_button.disabled = True

    try:
        with dashboard_output:
            clear_output(wait=True)

            print(
                f"Running batch size {batch_slider.value} "
                f"for approximately {duration_slider.value} seconds..."
            )

        summary, samples_df = run_training_experiment(
            batch_size=batch_slider.value,
            duration_seconds=float(duration_slider.value),
        )

        RUN_SUMMARIES.append(summary)
        RUN_TRACES[len(RUN_SUMMARIES) - 1] = samples_df

        with dashboard_output:
            clear_output(wait=True)
            render_dashboard(summary, samples_df)

            display(
                HTML(
                    """
                    <h3 style="font-family:Arial,sans-serif;">
                        Comparison with previous runs
                    </h3>
                    """
                )
            )
            render_comparison()

    finally:
        run_button.disabled = False
        clear_button.disabled = False


def on_clear_clicked(_):
    RUN_SUMMARIES.clear()
    RUN_TRACES.clear()

    with dashboard_output:
        clear_output(wait=True)
        print("Results cleared. Choose a batch size and run again.")


run_button.on_click(on_run_clicked)
clear_button.on_click(on_clear_clicked)

controls = widgets.VBox(
    [
        batch_slider,
        duration_slider,
        widgets.HBox([run_button, clear_button]),
    ]
)

display(controls, dashboard_output)

with dashboard_output:
    print("Choose a batch size and press Run Experiment.")

Output()

## 5. What should students notice?

### Small batches

- each step is fast;
- few samples are completed per step;
- kernel launches may be too short to keep the GPU continuously busy;
- utilization readings may jump up and down;
- throughput is usually low.

### Medium batches

- more samples share the same matrix operations;
- utilization rises;
- throughput can rise dramatically;
- the step takes longer, but not in proportion to the additional samples.

### Large batches

- throughput eventually plateaus;
- GPU utilization may already be near its ceiling;
- memory use continues to rise;
- another doubling may bring little benefit;
- eventually the batch may not fit.

Ask:

> At which batch size did this particular GPU stop gaining much throughput?

That answer is hardware- and model-specific.

# Part B — Inspect a Real CPU/GPU Timeline

The dashboard shows sampled hardware metrics.

PyTorch Profiler shows a different view:

- Python and CPU operations;
- CUDA kernels;
- forward operations;
- backward operations;
- optimizer operations;
- tensor shapes;
- time spent in each operation.

This section creates a Chrome-compatible trace file.

The trace can be opened in:

- `chrome://tracing`
- Perfetto's trace viewer
- TensorBoard's profiler interface

Profiling adds overhead, so profiler timing should not replace the benchmark timing above.

In [25]:
from torch.profiler import (
    profile,
    ProfilerActivity,
    record_function,
)


def create_profiler_trace(
    batch_size=256,
    steps=8,
    output_file="batch_training_trace.json",
):
    """
    Profile a small number of complete training steps.

    Returns the absolute trace-file path.
    """
    model = StressMLP().to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_function = nn.CrossEntropyLoss()

    # Warm-up before profiling.
    for step in range(4):
        start = step * batch_size
        indices = torch.arange(
            start,
            start + batch_size,
            device=DEVICE,
        ) % len(X_SYNTHETIC)

        optimizer.zero_grad(set_to_none=True)
        loss = loss_function(
            model(X_SYNTHETIC[indices]),
            Y_SYNTHETIC[indices],
        )
        loss.backward()
        optimizer.step()

    synchronize_cuda()

    with profile(
        activities=[
            ProfilerActivity.CPU,
            ProfilerActivity.CUDA,
        ],
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as profiler:

        for step in range(steps):
            start = (step + 4) * batch_size
            indices = torch.arange(
                start,
                start + batch_size,
                device=DEVICE,
            ) % len(X_SYNTHETIC)

            with record_function("TRAINING_STEP"):
                optimizer.zero_grad(set_to_none=True)

                with record_function("FORWARD_PASS"):
                    logits = model(X_SYNTHETIC[indices])
                    loss = loss_function(
                        logits,
                        Y_SYNTHETIC[indices],
                    )

                with record_function("BACKWARD_PASS"):
                    loss.backward()

                with record_function("OPTIMIZER_STEP"):
                    optimizer.step()

    trace_path = Path(output_file).resolve()
    profiler.export_chrome_trace(str(trace_path))

    print(
        profiler.key_averages().table(
            sort_by="self_cuda_time_total",
            row_limit=20,
        )
    )

    del model
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()

    return trace_path


TRACE_PATH = create_profiler_trace(
    batch_size=256,
    steps=8,
)

print("\nTrace created:", TRACE_PATH)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           FORWARD_PASS         0.00%       0.000us         0.00%       0.000us       0.000us      27.261ms        40.41%      27.261ms       3.408ms           0 B           0 B           0 B           0 

## How to present the trace

1. Run the profiler cell.
2. Locate `batch_training_trace.json`.
3. Open Chrome or Edge.
4. Navigate to `chrome://tracing`.
5. Load the JSON file.
6. Zoom into one `TRAINING_STEP`.

Point out:

- CPU work and CUDA work happen on separate tracks.
- A layer becomes one or more GPU kernels.
- Forward and backward passes are composed of many operations.
- Backpropagation is real GPU work, not one magical “backward” command.
- Small gaps can reveal launch overhead or CPU-side preparation.
- Larger batches usually make kernels longer and more substantial.

For a sharper comparison, create two traces:

```python
create_profiler_trace(batch_size=8, output_file="batch_8_trace.json")
create_profiler_trace(batch_size=1024, output_file="batch_1024_trace.json")
```

Then compare them side by side.

# Optional External Monitor

While the notebook runs, open a second PowerShell window and use:

```powershell
nvidia-smi -l 1
```

For selected fields:

```powershell
nvidia-smi --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw --format=csv -l 1
```

This is useful because students can see that the monitoring values come from the GPU driver, not from a custom educational animation.

On Windows, Task Manager can also show the GPU's Compute or CUDA graph, depending on the driver and Windows version.

# Teaching Summary

The strongest local-machine sequence is:

1. Run batch size 1.
2. Watch low or unstable utilization.
3. Run batch size 32.
4. Run batch size 256.
5. Run batch size 1024.
6. Observe when throughput stops growing strongly.
7. Open the profiler trace.
8. Show that the GPU executes kernels, not “images.”
9. Return to the batch-gradient notebook and connect hardware parallelism to gradient averaging.

The central conclusion:

> A larger batch gives the GPU more independent work to perform together. This raises throughput while the hardware is under-utilized. Once the GPU is saturated, larger batches mostly increase memory use and step duration.

# Exercise

Run three batch sizes and answer:

1. Which batch size had the highest throughput?
2. Which batch size first exceeded 90% mean utilization?
3. Did doubling batch size double samples per second?
4. At what point did throughput begin to plateau?
5. How much additional memory was required?

In [26]:
# YOUR CODE HERE

# Suggested batch sizes:
# 8, 128, 1024

In [27]:
# SOLUTION

# Use the interactive dashboard above:
#
# 1. Select batch size 8 and run.
# 2. Select batch size 128 and run.
# 3. Select batch size 1024 and run.
# 4. Compare the generated table and plots.
#
# There is no universal numeric answer.
# The result depends on the GPU, driver, PyTorch version, model, and workload.